# Tesla & GameStop: Stock and Revenue Analysis

**Final Project — Python, yfinance, Web Scraping, and Interactive Dashboards**

## Project objective
Retrieve historical share-price data for Tesla (TSLA) and GameStop (GME), collect reported revenue data from public web pages, and visualize each company's share price alongside its revenue.

### Questions covered
1. Tesla stock data — reset index, save `tesla_data`, and display `head()`.
2. Tesla revenue data — create `tesla_revenue` and display `tail()`.
3. GameStop stock data — reset index, save `gme_data`, and display `head()`.
4. GameStop revenue data — create `gme_revenue` and display `tail()`.
5. Tesla stock and revenue dashboard using `make_graph()`.
6. GameStop stock and revenue dashboard using `make_graph()`.

**How to run:** Run the notebook from top to bottom in Google Colab or Jupyter. Data is retrieved at runtime, so values and dates may differ from course screenshots. Some websites restrict automated scraping; revenue cells report errors rather than inventing values if a source is inaccessible.

## 0. Install and import libraries

In [ ]:
%pip -q install yfinance pandas requests beautifulsoup4 lxml plotly

In [ ]:
import re
import warnings
from io import StringIO

import pandas as pd
import requests
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
}
print('Libraries imported successfully.')

## Helper function — scrape reported revenue

The function tries a company's Macrotrends revenue page first, then Stock Analysis as a fallback. It searches returned HTML tables for a likely revenue table and standardizes the result to `Date` and `Revenue` columns where possible. Web pages can change their layout or block automated requests; inspect the returned table before using it in a report.

In [ ]:
def scrape_revenue(company, ticker):
    slug = company.lower().replace(' ', '-')
    sources = [
        f'https://www.macrotrends.net/stocks/charts/{ticker}/{slug}/revenue',
        f'https://stockanalysis.com/stocks/{ticker.lower()}/revenue/',
    ]
    errors = []
    for url in sources:
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            tables = pd.read_html(StringIO(response.text))
            if not tables:
                errors.append(f'{url}: no HTML tables found')
                continue
            selected = None
            for table in tables:
                headings = ' '.join(str(c).lower() for c in table.columns)
                sample = ' '.join(str(v).lower() for v in table.head(5).astype(str).values.flatten())
                if 'revenue' in headings or 'revenue' in sample:
                    selected = table.copy()
                    break
            if selected is None:
                selected = tables[0].copy()
            selected.columns = [str(c).strip() for c in selected.columns]
            date_candidates = [c for c in selected.columns if any(k in c.lower() for k in ['date','year','period','quarter','fiscal'])]
            revenue_candidates = [c for c in selected.columns if 'revenue' in c.lower()]
            if not revenue_candidates and len(selected.columns) >= 2:
                revenue_candidates = [selected.columns[1]]
            if date_candidates and revenue_candidates:
                selected = selected.rename(columns={date_candidates[0]:'Date', revenue_candidates[0]:'Revenue'})
            selected['Source'] = url
            return selected.reset_index(drop=True)
        except Exception as exc:
            errors.append(f'{url}: {type(exc).__name__}: {exc}')
    raise RuntimeError('Could not retrieve revenue data for ' + company + '.\n' + '\n'.join(errors) + '\nCheck whether the site blocks scraping or has changed its table structure. Do not fabricate values.')

## Question 1 — Use yfinance to extract Tesla stock data

Create a ticker object, retrieve the maximum available historical period, reset the index, save the dataframe, and display the first five rows.

In [ ]:
tesla = yf.Ticker('TSLA')
tesla_data = tesla.history(period='max')
tesla_data.reset_index(inplace=True)
tesla_data.to_csv('tesla_data.csv', index=False)
display(tesla_data.head())

## Question 2 — Use web scraping to extract Tesla revenue data

Create `tesla_revenue` and display its last five rows. Verify that the selected table contains revenue data.

In [ ]:
tesla_revenue = scrape_revenue('Tesla', 'TSLA')
display(tesla_revenue.tail())

## Question 3 — Use yfinance to extract GameStop stock data

Retrieve the maximum available historical period, reset the index, save the dataframe, and display the first five rows.

In [ ]:
gme = yf.Ticker('GME')
gme_data = gme.history(period='max')
gme_data.reset_index(inplace=True)
gme_data.to_csv('gme_data.csv', index=False)
display(gme_data.head())

## Question 4 — Use web scraping to extract GameStop revenue data

Create `gme_revenue` and display its last five rows.

In [ ]:
gme_revenue = scrape_revenue('GameStop', 'GME')
display(gme_revenue.tail())

## Dashboard helper — `make_graph()`

The function creates a two-panel interactive dashboard: historical closing share price above and reported revenue below. Revenue pages may report annual or quarterly values, so the lower panel uses the reporting periods supplied by the source.

In [ ]:
def _find_date_column(df):
    for col in df.columns:
        if str(col).strip().lower() in {'date','datetime','year','period','quarter','fiscal year'}:
            return col
    return df.columns[0]

def _numeric_revenue(series):
    def convert(value):
        if pd.isna(value):
            return float('nan')
        s = str(value).strip().replace(',', '').replace('$', '').replace('€', '').replace('£', '')
        multiplier = 1.0
        if s.endswith(('B','b')):
            multiplier, s = 1e9, s[:-1]
        elif s.endswith(('M','m')):
            multiplier, s = 1e6, s[:-1]
        elif s.endswith(('K','k')):
            multiplier, s = 1e3, s[:-1]
        s = re.sub(r'[^0-9.\\-]', '', s)
        try:
            return float(s) * multiplier
        except ValueError:
            return float('nan')
    return series.map(convert)

def make_graph(stock_data, revenue_data, stock):
    stock_df = stock_data.copy()
    date_col = _find_date_column(stock_df)
    stock_df[date_col] = pd.to_datetime(stock_df[date_col], errors='coerce')
    stock_df['Close'] = pd.to_numeric(stock_df['Close'], errors='coerce')
    stock_df = stock_df.dropna(subset=[date_col, 'Close']).sort_values(date_col)

    revenue_df = revenue_data.copy()
    rev_date_col = 'Date' if 'Date' in revenue_df.columns else _find_date_column(revenue_df)
    rev_value_col = 'Revenue' if 'Revenue' in revenue_df.columns else next((c for c in revenue_df.columns if 'revenue' in str(c).lower()), None)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.16,
                        subplot_titles=(f'{stock} Historical Share Price', f'{stock} Reported Revenue'))
    fig.add_trace(go.Scatter(x=stock_df[date_col], y=stock_df['Close'], mode='lines', name='Closing Share Price'), row=1, col=1)

    if rev_value_col is not None and not revenue_df.empty:
        revenue_df['Revenue_numeric'] = _numeric_revenue(revenue_df[rev_value_col])
        revenue_df['_period_label'] = revenue_df[rev_date_col].astype(str)
        revenue_df = revenue_df.dropna(subset=['Revenue_numeric'])
        fig.add_trace(go.Bar(x=revenue_df['_period_label'], y=revenue_df['Revenue_numeric'], name='Reported Revenue'), row=2, col=1)
        fig.update_yaxes(title_text='Revenue (as reported)', row=2, col=1)
    else:
        fig.add_annotation(text='No recognizable numeric revenue column found.', xref='paper', yref='paper', x=0.5, y=0.15, showarrow=False)

    fig.update_layout(title_text=f'{stock}: Stock Price and Revenue Dashboard', title_x=0.5,
                      template='plotly_white', height=800, hovermode='x unified')
    fig.update_xaxes(title_text='Date', row=1, col=1)
    fig.update_yaxes(title_text='Share Price (USD)', row=1, col=1)
    fig.update_xaxes(title_text='Reporting Period', row=2, col=1)
    return fig

## Question 5 — Tesla stock and revenue graph

Call `make_graph()` to display the Tesla dashboard.

In [ ]:
tesla_dashboard = make_graph(tesla_data, tesla_revenue, 'Tesla')
tesla_dashboard.show()

## Question 6 — GameStop stock and revenue graph

Call `make_graph()` to display the GameStop dashboard.

In [ ]:
gme_dashboard = make_graph(gme_data, gme_revenue, 'GameStop')
gme_dashboard.show()

## Final checks and submission

- Confirm `tesla_data.head()` and `gme_data.head()` display five rows.
- Confirm `tesla_revenue.tail()` and `gme_revenue.tail()` display the final five rows.
- Confirm both dashboards render with titles and labeled axes.
- Save the notebook after running it and take screenshots of the actual code and output cells requested by the assignment.
- If a revenue scrape fails, check the source site's access restrictions or table structure and rerun the relevant cell; do not fabricate data.

The stock-data cells save `tesla_data.csv` and `gme_data.csv` in the current working directory.